# M1-B1 — Exploration de l'historique des réservations

Ce notebook sert aux tâches 1 à 3 : connexion à la base, exploration en SQL, chargement des données.
Crée d'autres notebooks pour la suite (par exemple `02_modelisation.ipynb`).

In [2]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # pour pouvoir importer model.train_baseline plus tard
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

## Tâche 1 — La base répond-elle ?

In [3]:
pd.read_sql(text("SELECT COUNT(*) AS nb_reservations FROM reservations"), engine)

,nb_reservations
0,78703


## Tâche 2 — Explorer en SQL

Les calculs se font **dans la requête SQL** ; pandas sert seulement à afficher le résultat.

Pour commencer : les tables disponibles et leurs colonnes.

In [4]:
pd.read_sql(text("SELECT * FROM hotels"), engine)

,hotel_id,code,nom,type,ville
0,1,City Hotel,Escale Atlantique Centre,urbain,Lisbonne
1,2,Resort Hotel,Escale Atlantique Océan,resort,Algarve


In [7]:
pd.read_sql(text("SELECT * FROM reservations LIMIT 5"), engine)

,reservation_id,hotel_id,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date
0,1,1,0,6,2015,July,27,1,0,2,...,6,None,0,Transient,0.0,0,0,Check-Out,2015-07-03,2015-07-01
1,2,1,1,88,2015,July,27,1,0,4,...,9,None,0,Transient,76.5,0,1,Canceled,2015-07-01,2015-07-01
2,3,1,1,65,2015,July,27,1,0,4,...,9,None,0,Transient,68.0,0,1,Canceled,2015-04-30,2015-07-01
3,4,1,1,92,2015,July,27,1,2,4,...,9,None,0,Transient,76.5,0,2,Canceled,2015-06-23,2015-07-01
4,5,1,1,257,2015,July,27,1,0,2,...,6,None,0,Transient,101.5,0,0,Canceled,2015-06-16,2015-07-01


### Réservations et taux d'annulation par hôtel

In [18]:
query = text("""
    SELECT nom, count(reservation_id) as nb_reservations, sum(is_canceled) as nb_annulations, avg(is_canceled) as taux_annulation
    FROM hotels
    JOIN
    reservations ON hotels.hotel_id = reservations.hotel_id
    group by hotels.hotel_id, nom
""")
pd.read_sql(query, engine)

,nom,nb_reservations,nb_annulations,taux_annulation
0,Escale Atlantique Océan,26881,7068,0.262937
1,Escale Atlantique Centre,51822,21411,0.413164


### Taux d'annulation selon le mois d'arrivée

In [27]:
query = text("""
    SELECT arrival_date_month,count(reservation_id) as nb_reservations, sum(is_canceled) as nb_annulations, avg(is_canceled) as taux_annulation
    FROM reservations
    group by arrival_date_month
""")
pd.read_sql(query, engine)

,arrival_date_month,nb_reservations,nb_annulations,taux_annulation
0,June,5292,2096,0.396070
1,November,6794,2122,0.312334
2,March,4824,1477,0.306177
3,August,8952,3423,0.382373
4,December,6780,2371,0.349705
5,January,2248,557,0.247776
6,July,7348,2758,0.375340
7,April,5428,2061,0.379698
8,May,5478,1915,0.349580
9,February,3891,1337,0.343613


### Taux d'annulation selon le délai entre réservation et arrivée

In [22]:
query = text("""
    SELECT
        CASE
            WHEN lead_time < 7   THEN '1. moins de 7 jours'
            WHEN lead_time < 30  THEN '2. 7 à 29 jours'
            WHEN lead_time < 90  THEN '3. 30 à 89 jours'
            WHEN lead_time < 180 THEN '4. 90 à 179 jours'
            WHEN lead_time < 365 THEN '5. 180 à 364 jours'
            ELSE                      '6. 365 jours et plus'
        END AS tranche_delai,
        COUNT(*) AS nb_reservations,
        SUM(is_canceled) AS nb_annulations,
        ROUND(AVG(is_canceled), 3) AS taux_annulation
    FROM reservations
    GROUP BY tranche_delai
    ORDER BY tranche_delai
""")
pd.read_sql(query, engine)

,tranche_delai,nb_reservations,nb_annulations,taux_annulation
0,1. moins de 7 jours,13049,1262,0.097
1,2. 7 à 29 jours,13016,3298,0.253
2,3. 30 à 89 jours,20420,7286,0.357
3,4. 90 à 179 jours,15734,6804,0.432
4,5. 180 à 364 jours,14638,8392,0.573
5,6. 365 jours et plus,1846,1437,0.778


### Taux d'annulation par segment de clientèle et par type d'acompte

In [30]:
query = text("""
    SELECT
        market_segment,
        COUNT(*)                   AS nb_reservations,
        SUM(is_canceled)           AS nb_annulations,
        ROUND(AVG(is_canceled), 3) AS taux_annulation
    FROM reservations
    GROUP BY market_segment
    ORDER BY taux_annulation DESC
""")
pd.read_sql(query, engine)


,market_segment,nb_reservations,nb_annulations,taux_annulation
0,Undefined,2,2,1.000
1,Groups,13957,8695,0.623
2,Offline TA/TO,18552,6397,0.345
3,Online TA,33826,11482,0.339
4,Aviation,127,29,0.228
5,Corporate,3733,594,0.159
6,Direct,7977,1218,0.153
7,Complementary,529,62,0.117


In [31]:
query = text("""
    SELECT
        deposit_type,
        COUNT(*)                   AS nb_reservations,
        SUM(is_canceled)           AS nb_annulations,
        ROUND(AVG(is_canceled), 3) AS taux_annulation
    FROM reservations
    GROUP BY deposit_type
    ORDER BY taux_annulation DESC
""")
pd.read_sql(query, engine)

,deposit_type,nb_reservations,nb_annulations,taux_annulation
0,Non Refund,10461,10376,0.992
1,No Deposit,68107,18093,0.266
2,Refundable,135,10,0.074


### Rapport d'étonnement

Le segment groups annule à 62% ça me parrait très élevé, mais je ne sais pas à quoi "groups"" correspond dans la réalité métier.
Un tot d'annulation à 99% sur un dépot non remboursable est hautement suspect.


## Tâche 3 — Charger les données et préparer le découpage

Une seule requête SQL, avec la jointure sur `hotels` pour récupérer la colonne `hotel`
(`'City Hotel'` / `'Resort Hotel'`) attendue par la recette de la v1.

In [36]:
query = text("""
    SELECT reservations.*, hotels.code AS hotel
    FROM reservations
    JOIN hotels ON hotels.hotel_id = reservations.hotel_id
""")
df = pd.read_sql(query, engine, parse_dates=["arrival_date"])
df.head()

,reservation_id,hotel_id,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,hotel
0,1,1,0,6,2015,July,27,1,0,2,...,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-03,2015-07-01,City Hotel
1,2,1,1,88,2015,July,27,1,0,4,...,NaN,0,Transient,76.5,0,1,Canceled,2015-07-01,2015-07-01,City Hotel
2,3,1,1,65,2015,July,27,1,0,4,...,NaN,0,Transient,68.0,0,1,Canceled,2015-04-30,2015-07-01,City Hotel
3,4,1,1,92,2015,July,27,1,2,4,...,NaN,0,Transient,76.5,0,2,Canceled,2015-06-23,2015-07-01,City Hotel
4,5,1,1,257,2015,July,27,1,0,2,...,NaN,0,Transient,101.5,0,0,Canceled,2015-06-16,2015-07-01,City Hotel


In [37]:
date_limite = df["arrival_date"].max() - pd.DateOffset(months=4)
print("Date limite :", date_limite.date())

train = df[df["arrival_date"] <= date_limite]
print(f"train : {len(train)} réservations, du {train['arrival_date'].min().date()} au {train['arrival_date'].max().date()}")

test = df[df["arrival_date"] > date_limite]
print(f"test  : {len(test)} réservations, du {test['arrival_date'].min().date()} au {test['arrival_date'].max().date()}")


Date limite : 2016-08-31
train : 58792 réservations, du 2015-07-01 au 2016-08-31
test  : 19911 réservations, du 2016-09-01 au 2016-12-31


In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

y = train["is_canceled"]
X = train.drop(columns=[
    "is_canceled",
    "reservation_id",
    "arrival_date",
    "reservation_status_date",
    "reservation_status"
])
X = pd.get_dummies(X)

foret = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

scores = cross_val_score(foret, X, y, cv=5, scoring="accuracy", n_jobs=-1)
print(scores, scores.mean())

foret.fit(X, y)
importances = pd.Series(foret.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(15)

[0.79845225 0.86682541 0.81884674 0.87608437 0.78414696] 0.8288711468334806


lead_time                      0.092306
country_PRT                    0.076141
deposit_type_Non Refund        0.072982
deposit_type_No Deposit        0.062648
previous_cancellations         0.062106
adr                            0.061885
total_of_special_requests      0.053403
agent                          0.044581
arrival_date_day_of_month      0.041327
arrival_date_week_number       0.040242
stays_in_week_nights           0.031047
stays_in_weekend_nights        0.020782
required_car_parking_spaces    0.018321
assigned_room_type_A           0.017711
booking_changes                0.016079
dtype: float64